# 02 Train and Validate R-SCoLQ

**R-SCoLQ = Round-Stable Source-Calibrated Localization Quality**.

01のSCoLQは「このbbox単体がsource GT上で局所化品質を持つか」を学習した。02では、08_3_6の結果で見えた `mean_scolq` と `pseudo_total` の同時増加を明示的に罰し、Phase2を何ラウンドも回したときに自己強化しにくい報酬へ変える。

重要な切り分け:

- source GTは、static bbox quality model の学習に使う。
- 08_3_6 の target evaluation は、R-SCoLQ policy が late drift を検知できるかの診断にだけ使う。
- 保存する `rscolq_best.joblib` の policy には target GT 由来の選択を入れない。

## Design

R-SCoLQは2段に分ける。

```text
static_score = source GTで学習したbbox品質
round_multiplier = pseudo label inflation / quality inflation / class budget drift への罰則

R-SCoLQ = static_score * round_multiplier * class_multiplier
```

今回のポイントは、`conf_logit`, `rank_conf`, `split_pred_class_prior` のようなモデルが自己学習で作り替えやすい特徴を強く使わないこと。source validation精度だけでなく、長期報酬として危ない特徴量には proxy penalty を入れて選ぶ。

In [1]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "dynamic_quality_aware_classwise_aggregation").exists() and (path / "navigating_data_heterogeneity").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
DQA_ROOT = REPO_ROOT / "dynamic_quality_aware_classwise_aggregation"
SCOLQ_ROOT = DQA_ROOT / "source_calibrated_localization_quality"
ARTIFACT_ROOT = SCOLQ_ROOT / "artifacts"
REPORT_ROOT = SCOLQ_ROOT / "reports"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_DATASET = ARTIFACT_ROOT / "scolq_dataset_with_agreement.csv"
SOURCE_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_scene_tri_stage_policy_8h"
SOURCE_TRAIN_LIST = SOURCE_WORK_ROOT / "data_lists" / "server_cloudy_train.txt"
SOURCE_VAL_LIST = SOURCE_WORK_ROOT / "data_lists" / "server_cloudy_val.txt"
PHASE2_STATS_ROOT = DQA_ROOT / "stats_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003"
PHASE2_WORK_ROOT = DQA_ROOT / "efficientteacher_dqa08_3_6_phase2_scolq_policy" / "a_scolq_soft_bbox_r003"
EVAL_EARLY_TOTAL = PHASE2_WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary_0836_early_total.csv"
EVAL_ALL = PHASE2_WORK_ROOT / "validation_reports" / "paper_protocol_eval_summary.csv"

RANDOM_SEED = 738
TRAIN_SAMPLE = 350_000
VAL_SAMPLE = 214_589  # source_val is about this size; keep all by default.

print("repo:", REPO_ROOT)
print("source_dataset:", SOURCE_DATASET, SOURCE_DATASET.exists())
print("source_train_list:", SOURCE_TRAIN_LIST, SOURCE_TRAIN_LIST.exists())
print("phase2_stats:", PHASE2_STATS_ROOT, PHASE2_STATS_ROOT.exists())
print("phase2_eval:", EVAL_EARLY_TOTAL, EVAL_EARLY_TOTAL.exists())

repo: /app/Object_Detection
source_dataset: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/scolq_dataset_with_agreement.csv True
source_train_list: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_scene_tri_stage_policy_8h/data_lists/server_cloudy_train.txt True
phase2_stats: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/stats_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003 True
phase2_eval: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/efficientteacher_dqa08_3_6_phase2_scolq_policy/a_scolq_soft_bbox_r003/validation_reports/paper_protocol_eval_summary_0836_early_total.csv True


In [2]:
STATIC_BASE_FEATURES = [
    "cls",
    "x", "y", "w", "h",
    "area", "log_area", "aspect", "log_aspect", "edge_dist",
    "max_iou_same_pred", "max_iou_any_pred",
    "near_same_count_50", "near_any_count_50",
    "source_gt_class_prior", "source_gt_class_log_prior",
    "aug640_matched", "plain512_matched", "plain768_matched", "agreement_match_count",
]
CONTEXT_FEATURES = ["image_pred_count", "class_pred_count"]

FEATURE_SETS = {
    # safest, but sometimes too weak: no confidence-like signal at all.
    "safe_no_conf": STATIC_BASE_FEATURES + CONTEXT_FEATURES,
    # default R-SCoLQ candidate: weak confidence is allowed, but no logit/rank/prior-ratio shortcut.
    "low_conf": STATIC_BASE_FEATURES + CONTEXT_FEATURES + ["conf"],
    # ablation: adds rank features, which are useful but easier for self-training to game.
    "rank_conf": STATIC_BASE_FEATURES + CONTEXT_FEATURES + ["conf", "rank_conf", "rank_conf_norm"],
    # SCoLQ-like proxy-heavy candidate. This is expected to score well on source but is risky long-term.
    "all_proxy": STATIC_BASE_FEATURES + CONTEXT_FEATURES + [
        "conf", "rank_conf", "rank_conf_norm", "conf_logit", "split_pred_class_prior", "pred_gt_prior_ratio"
    ],
}

HIGH_RISK_FEATURES = {"conf_logit", "rank_conf", "rank_conf_norm", "split_pred_class_prior", "pred_gt_prior_ratio"}
MED_RISK_FEATURES = {"conf", "image_pred_count", "class_pred_count"}

LABEL_COLS = ["split", "good50", "good75", "max_iou_any_gt"]
USECOLS = sorted(set(LABEL_COLS + sum(FEATURE_SETS.values(), [])))
USECOLS

['agreement_match_count',
 'area',
 'aspect',
 'aug640_matched',
 'class_pred_count',
 'cls',
 'conf',
 'conf_logit',
 'edge_dist',
 'good50',
 'good75',
 'h',
 'image_pred_count',
 'log_area',
 'log_aspect',
 'max_iou_any_gt',
 'max_iou_any_pred',
 'max_iou_same_pred',
 'near_any_count_50',
 'near_same_count_50',
 'plain512_matched',
 'plain768_matched',
 'pred_gt_prior_ratio',
 'rank_conf',
 'rank_conf_norm',
 'source_gt_class_log_prior',
 'source_gt_class_prior',
 'split',
 'split_pred_class_prior',
 'w',
 'x',
 'y']

In [3]:
def load_source_quality_dataset(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, usecols=USECOLS)
    train = df[df["split"].eq("source_train")].copy()
    val = df[df["split"].eq("source_val")].copy()
    if len(train) > TRAIN_SAMPLE:
        train = train.sample(n=TRAIN_SAMPLE, random_state=RANDOM_SEED)
    if len(val) > VAL_SAMPLE:
        val = val.sample(n=VAL_SAMPLE, random_state=RANDOM_SEED + 1)
    return train.reset_index(drop=True), val.reset_index(drop=True)


train_df, val_df = load_source_quality_dataset(SOURCE_DATASET)
print("train:", train_df.shape, "val:", val_df.shape)
display(pd.concat([
    train_df.assign(part="train"),
    val_df.assign(part="val"),
]).groupby("part")[["good50", "good75", "max_iou_any_gt"]].mean())

train: (350000, 32) val: (214589, 32)


,good50,good75,max_iou_any_gt
part,,,
train,0.090649,0.033211,0.422330
val,0.091384,0.032234,0.415201


In [4]:
def make_pipeline(features: list[str]) -> Pipeline:
    cat = ["cls"] if "cls" in features else []
    num = [f for f in features if f not in cat]
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    pre = ColumnTransformer(
        [
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", encoder, cat),
        ],
        sparse_threshold=0.0,
    )
    clf = HistGradientBoostingClassifier(
        max_iter=140,
        learning_rate=0.06,
        l2_regularization=0.04,
        max_leaf_nodes=31,
        random_state=RANDOM_SEED,
    )
    return Pipeline([("pre", pre), ("clf", clf)])


def quality_metrics(y50: np.ndarray, y75: np.ndarray, iou: np.ndarray, score: np.ndarray) -> dict[str, float]:
    out: dict[str, float] = {
        "ap50_quality": float(average_precision_score(y50, score)),
        "ap75_quality": float(average_precision_score(y75, score)),
        "brier50": float(brier_score_loss(y50, np.clip(score, 1e-6, 1.0 - 1e-6))),
    }
    try:
        out["roc50"] = float(roc_auc_score(y50, score))
    except Exception:
        out["roc50"] = float("nan")
    order = np.argsort(-score)
    for pct in [0.05, 0.10, 0.20, 0.40]:
        k = max(1, int(len(score) * pct))
        idx = order[:k]
        key = int(pct * 100)
        out[f"p50_at_{key}pct"] = float(np.mean(y50[idx]))
        out[f"p75_at_{key}pct"] = float(np.mean(y75[idx]))
        out[f"mean_iou_at_{key}pct"] = float(np.mean(iou[idx]))
    return out


def proxy_penalty(features: list[str]) -> float:
    return 0.05 * len(set(features) & HIGH_RISK_FEATURES) + 0.015 * len(set(features) & MED_RISK_FEATURES)


def selection_score(metrics: dict[str, float], features: list[str]) -> float:
    return float(
        metrics["ap50_quality"]
        + metrics["ap75_quality"]
        + 0.8 * metrics["p50_at_10pct"]
        + 0.5 * metrics["p75_at_10pct"]
        + 0.5 * metrics["mean_iou_at_10pct"]
        - proxy_penalty(features)
    )

In [5]:
y_train = train_df["good50"].astype(int).to_numpy()
pos = max(int(y_train.sum()), 1)
neg = max(int(len(y_train) - pos), 1)
sample_weight = np.where(y_train == 1, neg / pos, 1.0)

models: dict[str, dict] = {}
rows = []
for feature_set_name, raw_features in FEATURE_SETS.items():
    features = [f for f in raw_features if f in train_df.columns and (f == "cls" or train_df[f].nunique(dropna=True) > 1)]
    pipe = make_pipeline(features)
    pipe.fit(train_df[features], y_train, clf__sample_weight=sample_weight)
    score = pipe.predict_proba(val_df[features])[:, 1]
    metrics = quality_metrics(
        val_df["good50"].astype(int).to_numpy(),
        val_df["good75"].astype(int).to_numpy(),
        val_df["max_iou_any_gt"].astype(float).to_numpy(),
        score,
    )
    metrics["proxy_penalty"] = proxy_penalty(features)
    metrics["selection_score"] = selection_score(metrics, features)
    key = f"{feature_set_name}:hgb"
    rows.append({"candidate": key, "feature_set": feature_set_name, "n_features": len(features), **metrics})
    models[key] = {"pipeline": pipe, "features": features, "metrics": metrics}

ranking = pd.DataFrame(rows).sort_values("selection_score", ascending=False).reset_index(drop=True)
ranking.to_csv(REPORT_ROOT / "rscolq_model_ranking.csv", index=False)
best_key = str(ranking.loc[0, "candidate"])
print("best:", best_key)
display(ranking)

best: low_conf:hgb


,candidate,feature_set,n_features,ap50_quality,ap75_quality,brier50,roc50,p50_at_5pct,p75_at_5pct,mean_iou_at_5pct,...,p75_at_10pct,mean_iou_at_10pct,p50_at_20pct,p75_at_20pct,mean_iou_at_20pct,p50_at_40pct,p75_at_40pct,mean_iou_at_40pct,proxy_penalty,selection_score
0,low_conf:hgb,low_conf,19,0.767833,0.795002,0.083211,0.952460,0.874080,0.529779,0.724090,...,0.293457,0.586697,0.404921,0.155649,0.442534,0.223464,0.079758,0.334576,0.045,2.480720
1,rank_conf:hgb,rank_conf,21,0.768683,0.794828,0.083082,0.952745,0.875478,0.530804,0.724597,...,0.294156,0.586964,0.405154,0.155626,0.443435,0.223627,0.079746,0.334004,0.145,2.382401
2,all_proxy:hgb,all_proxy,24,0.768551,0.795472,0.083195,0.952599,0.874825,0.530245,0.724617,...,0.293876,0.586087,0.404711,0.155556,0.443033,0.223662,0.079758,0.335078,0.295,2.231998
3,safe_no_conf:hgb,safe_no_conf,18,0.614495,0.512432,0.116806,0.914016,0.717961,0.385963,0.653040,...,0.246388,0.544174,0.363213,0.142857,0.425312,0.214971,0.077299,0.338677,0.030,1.930348


## Round-Stability Diagnostics from 08_3_6

ここでは08_3_6の内部statsだけから round multiplier を作る。target評価のmAPは、あとで「この設計ならlate roundを危険側に落とせていたか」を見るための診断列として付けるだけ。

In [6]:
def load_phase2_round_stats(stats_root: Path) -> pd.DataFrame:
    rows = []
    for path in sorted(stats_root.glob("phase2_round[0-9][0-9][0-9].json")):
        data = json.loads(path.read_text())
        counts = np.zeros(10, dtype=np.float64)
        quality_sums = np.zeros(10, dtype=np.float64)
        objectness_sums = np.zeros(10, dtype=np.float64)
        class_conf_sums = np.zeros(10, dtype=np.float64)
        for client in data.get("clients", []):
            counts += np.asarray(client.get("counts", [0.0] * 10)[:10], dtype=np.float64)
            quality_sums += np.asarray(client.get("quality_sums", [0.0] * 10)[:10], dtype=np.float64)
            objectness_sums += np.asarray(client.get("objectness_sums", [0.0] * 10)[:10], dtype=np.float64)
            class_conf_sums += np.asarray(client.get("class_confidence_sums", [0.0] * 10)[:10], dtype=np.float64)
        total = float(counts.sum())
        if total <= 0:
            continue
        round_id = int(path.name.split("round", 1)[1].split(".", 1)[0])
        row = {
            "round": round_id,
            "pseudo_total": total,
            "mean_scolq": float(quality_sums.sum() / total),
            "mean_obj": float(objectness_sums.sum() / total),
            "mean_cls_conf": float(class_conf_sums.sum() / total),
        }
        row.update({f"class_{i}_count": float(counts[i]) for i in range(10)})
        rows.append(row)
    return pd.DataFrame(rows).sort_values("round").reset_index(drop=True)


def attach_target_eval_for_diagnostic(round_df: pd.DataFrame) -> pd.DataFrame:
    out = round_df.copy()
    eval_frames = []
    for path in [EVAL_EARLY_TOTAL, EVAL_ALL]:
        if not path.exists():
            continue
        eval_df = pd.read_csv(path)
        eval_df = eval_df[
            eval_df["split"].eq("scene_total")
            & eval_df["checkpoint_label"].str.contains("a_scolq_soft_bbox_r003_r", na=False)
        ].copy()
        if eval_df.empty:
            continue
        eval_df["round"] = eval_df["checkpoint_label"].str.extract(r"_r(\d+)$").astype(int)
        eval_frames.append(eval_df[["round", "map50", "map50_95"]].drop_duplicates("round"))
    if eval_frames:
        merged_eval = pd.concat(eval_frames).drop_duplicates("round")
        out = out.merge(merged_eval, on="round", how="left")
    return out


round_df = load_phase2_round_stats(PHASE2_STATS_ROOT)
round_df = attach_target_eval_for_diagnostic(round_df)
display(round_df)

,round,pseudo_total,mean_scolq,mean_obj,mean_cls_conf,class_0_count,class_1_count,class_2_count,class_3_count,class_4_count,class_5_count,class_6_count,class_7_count,class_8_count,class_9_count,map50,map50_95
0,1,770515.0,0.443193,0.344575,0.885031,86931.0,147.0,316177.0,3802.0,26084.0,3711.0,1.0,103519.0,230143.0,0.0,0.382,0.211
1,2,773621.0,0.448643,0.353537,0.888031,81923.0,133.0,316237.0,4199.0,28658.0,3945.0,0.0,103402.0,235124.0,0.0,0.381,0.212
2,3,757997.0,0.450761,0.353734,0.893407,87371.0,128.0,316108.0,3825.0,25320.0,3535.0,0.0,100409.0,221301.0,0.0,0.377,0.211
3,4,760731.0,0.453214,0.355148,0.894357,83274.0,122.0,315941.0,4100.0,24306.0,3191.0,0.0,103382.0,226415.0,0.0,NaN,NaN
4,5,770963.0,0.456014,0.357044,0.893967,88444.0,130.0,315865.0,3963.0,25496.0,3126.0,0.0,104628.0,229311.0,0.0,NaN,NaN
5,6,785763.0,0.460806,0.360953,0.892723,88819.0,139.0,316050.0,4179.0,25530.0,3303.0,0.0,110817.0,236926.0,0.0,NaN,NaN
6,7,795512.0,0.466490,0.365772,0.892233,90509.0,149.0,316491.0,4322.0,26672.0,2955.0,0.0,112750.0,241664.0,0.0,NaN,NaN
7,8,807088.0,0.470166,0.369221,0.890618,92436.0,136.0,316008.0,4538.0,28635.0,2723.0,0.0,113220.0,249392.0,0.0,NaN,NaN
8,9,817731.0,0.476864,0.375695,0.890034,94073.0,169.0,316611.0,4047.0,30442.0,2748.0,1.0,117122.0,252518.0,0.0,NaN,NaN
9,10,819653.0,0.487045,0.385868,0.889442,96931.0,167.0,316338.0,4283.0,31262.0,2803.0,2.0,114717.0,253150.0,0.0,0.357,0.196


In [7]:
def compute_round_stability_policy(round_df: pd.DataFrame, anchor_rounds: tuple[int, ...] = (1, 2)) -> tuple[pd.DataFrame, dict]:
    anchor = round_df[round_df["round"].isin(anchor_rounds)].copy()
    if anchor.empty:
        raise RuntimeError("Need at least one anchor round.")

    policy = {
        "anchor_rounds": list(anchor_rounds),
        "anchor_total": float(anchor["pseudo_total"].median()),
        "anchor_mean_scolq": float(anchor["mean_scolq"].median()),
        "anchor_class_counts": {
            str(i): float(max(anchor[f"class_{i}_count"].median(), 1.0)) for i in range(10)
        },
        "global_budget_margin": 0.01,
        "quality_margin": 0.005,
        "global_power": 3.0,
        "quality_power": 10.0,
        "class_power": 0.35,
        "min_multiplier": 0.15,
        "use_target_gt_for_policy": False,
    }

    rows = []
    for _, row in round_df.iterrows():
        pseudo_inflation = max(
            0.0,
            float(row["pseudo_total"]) / (policy["anchor_total"] * (1.0 + policy["global_budget_margin"])) - 1.0,
        )
        quality_inflation = max(
            0.0,
            float(row["mean_scolq"]) - policy["anchor_mean_scolq"] - policy["quality_margin"],
        )
        global_multiplier = math.exp(-policy["global_power"] * pseudo_inflation) * math.exp(
            -policy["quality_power"] * quality_inflation
        )
        class_multipliers = []
        for i in range(10):
            anchor_count = policy["anchor_class_counts"][str(i)]
            ratio = float(row[f"class_{i}_count"]) / (anchor_count * (1.0 + policy["global_budget_margin"]))
            if ratio > 1.0:
                mult = ratio ** (-policy["class_power"])
            else:
                mult = 1.0
            class_multipliers.append(float(np.clip(mult, policy["min_multiplier"], 1.0)))
        rows.append(
            {
                "round": int(row["round"]),
                "global_multiplier": float(global_multiplier),
                "mean_class_multiplier": float(np.mean(class_multipliers)),
                "min_class_multiplier": float(np.min(class_multipliers)),
                "rscolq_proxy": float(row["mean_scolq"] * global_multiplier * np.mean(class_multipliers)),
            }
        )
    return round_df.merge(pd.DataFrame(rows), on="round", how="left"), policy


round_diagnostic, round_policy = compute_round_stability_policy(round_df)
round_diagnostic.to_csv(REPORT_ROOT / "rscolq_0836_round_diagnostics.csv", index=False)
display(round_diagnostic[[
    "round", "pseudo_total", "mean_scolq", "global_multiplier", "mean_class_multiplier", "rscolq_proxy", "map50", "map50_95"
]])
round_policy

,round,pseudo_total,mean_scolq,global_multiplier,mean_class_multiplier,rscolq_proxy,map50,map50_95
0,1,770515.0,0.443193,1.000000,0.997977,0.442297,0.382,0.211
1,2,773621.0,0.448643,1.000000,0.996683,0.447155,0.381,0.212
2,3,757997.0,0.450761,1.000000,0.999152,0.450378,0.377,0.211
3,4,760731.0,0.453214,0.977300,0.999490,0.442700,NaN,NaN
4,5,770963.0,0.456014,0.950322,0.998685,0.432790,NaN,NaN
5,6,785763.0,0.460806,0.885269,0.995085,0.405932,NaN,NaN
6,7,795512.0,0.466490,0.805568,0.990184,0.372101,NaN,NaN
7,8,807088.0,0.470166,0.742671,0.987176,0.344701,NaN,NaN
8,9,817731.0,0.476864,0.666687,0.980792,0.311813,NaN,NaN
9,10,819653.0,0.487045,0.597721,0.956666,0.278502,0.357,0.196


{'anchor_rounds': [1, 2],
 'anchor_total': 772068.0,
 'anchor_mean_scolq': 0.44591814539354463,
 'anchor_class_counts': {'0': 84427.0,
  '1': 140.0,
  '2': 316207.0,
  '3': 4000.5,
  '4': 27371.0,
  '5': 3828.0,
  '6': 1.0,
  '7': 103460.5,
  '8': 232633.5,
  '9': 1.0},
 'global_budget_margin': 0.01,
 'quality_margin': 0.005,
 'global_power': 3.0,
 'quality_power': 10.0,
 'class_power': 0.35,
 'min_multiplier': 0.15,
 'use_target_gt_for_policy': False}

## Save Artifact

保存するartifactには、static model と round policy を入れる。`map50` / `map50_95` は診断レポートにだけ保存し、artifactのpolicy決定には入れない。

In [8]:
best = models[best_key]
artifact = {
    "name": "R-SCoLQ",
    "long_name": "Round-Stable Source-Calibrated Localization Quality",
    "candidate": best_key,
    "features": best["features"],
    "pipeline": best["pipeline"],
    "metrics": best["metrics"],
    "source_train_list": str(SOURCE_TRAIN_LIST),
    "source_val_list": str(SOURCE_VAL_LIST),
    "round_policy": round_policy,
    "diagnostic_sources": {
        "source_dataset": str(SOURCE_DATASET),
        "phase2_stats": str(PHASE2_STATS_ROOT),
        "target_eval_used_for_diagnostic_only": True,
    },
    "intended_use": {
        "static_score": "bbox loss / DQA quality base score",
        "round_multiplier": "preset anti-inflation penalty computed from pseudo stats",
        "class_multiplier": "class-wise pseudo budget guard",
        "next_notebook": "08_3_7_dqa_scene_phase2_round_stable_scolq_policy.ipynb",
    },
}

out_path = ARTIFACT_ROOT / "rscolq_best.joblib"
joblib.dump(artifact, out_path)
print("saved:", out_path)
print(json.dumps({k: artifact[k] for k in ["name", "candidate", "features", "metrics", "round_policy"]}, indent=2, ensure_ascii=False, default=float))

saved: /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/source_calibrated_localization_quality/artifacts/rscolq_best.joblib
{
  "name": "R-SCoLQ",
  "candidate": "low_conf:hgb",
  "features": [
    "cls",
    "x",
    "y",
    "w",
    "h",
    "area",
    "log_area",
    "aspect",
    "log_aspect",
    "edge_dist",
    "max_iou_same_pred",
    "max_iou_any_pred",
    "near_same_count_50",
    "near_any_count_50",
    "source_gt_class_prior",
    "source_gt_class_log_prior",
    "image_pred_count",
    "class_pred_count",
    "conf"
  ],
  "metrics": {
    "ap50_quality": 0.76783296713716,
    "ap75_quality": 0.7950022826832431,
    "brier50": 0.08321077931325967,
    "roc50": 0.9524597031682845,
    "p50_at_5pct": 0.8740795973529686,
    "p75_at_5pct": 0.5297791033647125,
    "mean_iou_at_5pct": 0.7240901566056281,
    "p50_at_10pct": 0.6535091807251375,
    "p75_at_10pct": 0.2934569857395843,
    "mean_iou_at_10pct": 0.5866970563376989,
    "p50_at_20pct": 0.40492112

## Reading

期待する読み方:

- `low_conf` が選ばれるなら、confidenceを完全に捨てるとsource品質が落ちすぎるが、logit/rank/prior-ratio系を足すほどの価値は薄い。
- `rscolq_proxy` が late round で下がるなら、08_3_6で見えた `mean_scolq` 増加とは逆に、R-SCoLQはpseudo label inflationを危険信号として扱えている。
- ここで保存したartifactはまだ実験設計用。次は08_3_7で、このround multiplierを実際のPhase2 trainingへ入れる。